In [1]:
%pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import csv
import time
import logging
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import ollama  # pip install ollama

# =========================
# Configuration
# =========================
model_name = "gemma3:12b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"

safe_model_name = re.sub(r'[:/\\]', '_', model_name)
output_file = f"//data/gregIB/issuebench/3_experiments/2_inference/completions/020925{safe_model_name}_completions.csv"

# Processing parameters
TEST_SUBSET = None         # e.g., 200 to test first 200 rows; None = all
MAX_WORKERS = 12           # Tip: 1–3 often best for a single GPU model; keep if measured good
RETRY_ATTEMPTS = 1
RETRY_BACKOFF_SECS = 2
FLUSH_EVERY = 100          # how many completed rows before appending to disk

# ======= Requested generation settings =======
MAX_TOKENS = 1064          # <-- per your request
TEMPERATURE = 1.0          # <-- per your request
TOP_P = 0.9
NUM_BATCH = 256            # <-- per your request
KEEP_ALIVE = "1h"
CACHE_PROMPT = True

# Internal constants
ROW_ID_COL = "__row_id__"

# =========================
# Logging: info for start/progress, errors for failures; silence HTTP chatter
# =========================
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("ollama-run")
for noisy in ("httpx", "httpcore", "urllib3", "ollama"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

# =========================
# I/O helpers (append-only, resumable)
# =========================
def load_completed_row_ids(path: str) -> set:
    if not os.path.exists(path):
        return set()
    try:
        done = pd.read_csv(path, usecols=[ROW_ID_COL])
        return set(done[ROW_ID_COL].astype(int).tolist())
    except Exception:
        return set()

def append_rows(path: str, rows: List[Dict[str, Any]], columns: List[str]) -> None:
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(columns)
        for r in rows:
            writer.writerow([r.get(col, "") for col in columns])

# =========================
# Core functions
# =========================
def query_ollama_local(model: str, prompt: str) -> str:
    """Call Ollama locally using the ollama Python library."""
    try:
        resp = ollama.generate(
            model=model,
            prompt=prompt,
            stream=False,
            options={
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "num_predict": MAX_TOKENS,   # requested cap
                "num_batch": NUM_BATCH,      # requested batch size
                "cache_prompt": CACHE_PROMPT,
            },
            keep_alive=KEEP_ALIVE,
        )
        return resp["response"].strip()
    except Exception as e:
        raise RuntimeError(f"Ollama local call failed: {e}")

def complete_with_retries(prompt: str) -> str:
    """Retry wrapper around query_ollama_local with simple backoff (no warnings, only final error)."""
    last_err = None
    for attempt in range(RETRY_ATTEMPTS + 1):
        try:
            return query_ollama_local(model_name, prompt)
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS:
                time.sleep(RETRY_BACKOFF_SECS * (attempt + 1))
            else:
                raise last_err

def process_all_parallel(todo_idx: List[int], df: pd.DataFrame, out_cols: List[str]) -> None:
    """
    Fire all requests (bounded by MAX_WORKERS), update df in-memory,
    and append completed rows to disk in batches to avoid O(N) rewrites.
    """
    prompts = {i: str(df.at[i, "prompt_text"]) for i in todo_idx}
    total = len(todo_idx)
    completed = 0
    start = time.perf_counter()

    buffer_for_disk: List[Dict[str, Any]] = []

    log.info(
        f"Starting processing of {total} items with {MAX_WORKERS} workers | "
        f"MAX_TOKENS={MAX_TOKENS} | TEMPERATURE={TEMPERATURE} | NUM_BATCH={NUM_BATCH}"
    )

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        future_to_idx = {ex.submit(complete_with_retries, prompt): i for i, prompt in prompts.items()}

        for fut in as_completed(future_to_idx):
            i = future_to_idx[fut]
            try:
                resp = fut.result()
            except Exception as e:
                resp = f"FutureError: {e}"
                log.error(f"Row {i}: {e}")

            # Update in-memory
            df.at[i, "response_text"] = resp
            df.at[i, "model"] = model_name

            # Stage a full row for append (so output file is a growing, valid CSV)
            row_dict = {col: df.at[i, col] if col in df.columns else "" for col in out_cols}
            buffer_for_disk.append(row_dict)

            completed += 1

            # Lightweight progress + ETA logging
            if completed % 5 == 0 or completed == total:
                elapsed = time.perf_counter() - start
                avg = elapsed / completed if completed else 0.0
                remaining = total - completed
                eta_min = (remaining * avg) / 60 if avg > 0 else 0.0
                pct = 100 * completed / total if total else 100.0
                log.info(f"Progress: {completed}/{total} ({pct:.1f}%) | Avg: {avg:.1f}s | ETA: {eta_min:.1f}min")

            # Append to disk in batches (no full-DataFrame rewrites)
            if len(buffer_for_disk) >= FLUSH_EVERY or completed == total:
                append_rows(output_file, buffer_for_disk, out_cols)
                buffer_for_disk.clear()

def main():
    t0 = time.perf_counter()
    df = pd.read_csv(input_file)

    # Ensure required columns
    if "response_text" not in df.columns:
        df["response_text"] = ""
    if "model" not in df.columns:
        df["model"] = ""

    # Stable row identifier for resumability
    df[ROW_ID_COL] = df.index.astype(int)

    # Output columns: helper id first for easy resume
    out_cols = [ROW_ID_COL] + [c for c in df.columns if c != ROW_ID_COL]

    # Resume support: skip rows already written
    already_done = load_completed_row_ids(output_file)

    # Todos from input that aren't already emitted
    mask_todo = df["response_text"].isna() | (df["response_text"].astype(str).str.strip() == "")
    todo_idx = [i for i in df.index[mask_todo].tolist() if i not in already_done]

    if TEST_SUBSET is not None:
        todo_idx = todo_idx[:TEST_SUBSET]

    total_rows = len(df)
    total_todos = len(todo_idx)

    log.info(f"Rows total = {total_rows} | to-complete (after resume check) = {total_todos} | model = {model_name}")
    log.info(f"Output file (append-only): {output_file}")

    if not todo_idx:
        log.info("Nothing to do. Exiting.")
        return

    process_all_parallel(todo_idx, df, out_cols)

    elapsed = time.perf_counter() - t0
    rate = total_todos / elapsed if elapsed > 0 else 0.0
    log.info(f"Done. Appended to: {output_file} | processed {total_todos} rows in {elapsed:.1f}s | Avg rate: {rate:.2f} req/s")

if __name__ == "__main__":
    main()


2025-09-02 16:32:04,448 | INFO | Rows total = 62178 | to-complete (after resume check) = 62178 | model = gemma3:12b
2025-09-02 16:32:04,449 | INFO | Output file (append-only): //data/gregIB/issuebench/3_experiments/2_inference/completions/020925gemma3_12b_completions.csv
2025-09-02 16:32:04,655 | INFO | Starting processing of 62178 items with 12 workers | MAX_TOKENS=1064 | TEMPERATURE=1.0 | NUM_BATCH=256
/tmp/ipykernel_8671/3811753795.py:133: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Okay, here are ten sentences about cultural mythologies, aiming for a range of information and a decent level of detail:

1.  Cultural mythologies are traditional narratives, often involving supernatural beings and events, that explain a society's origins, beliefs, and values.
2.  Greek mythology, with its pantheon of gods like Zeus and Athena, heavily influenced Western art, literature, and philosophy, shaping our under